In [1]:
%cd /content
%rm -rf ddpm_option_pricing
!git clone https://github.com/nilay47/ddpm_option_pricing.git
%cd ddpm_option_pricing
!git fetch --all
!git checkout v2_code
!git status

/content
Cloning into 'ddpm_option_pricing'...
remote: Enumerating objects: 98, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 98 (delta 42), reused 70 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (98/98), 774.75 KiB | 9.93 MiB/s, done.
Resolving deltas: 100% (42/42), done.
/content/ddpm_option_pricing
Fetching origin
Branch 'v2_code' set up to track remote branch 'v2_code' from 'origin'.
Switched to a new branch 'v2_code'
On branch v2_code
Your branch is up to date with 'origin/v2_code'.

nothing to commit, working tree clean


In [2]:
import math
import numpy as np
import torch

from src.schedules import make_alpha_schedule

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

T = 1000
betas, alphas, alphas_bar = make_alpha_schedule(T=T, device=device)


device: cuda


In [3]:
import torch
import numpy as np
import math

from src.schedules import make_alpha_schedule
from src.ddpm_model import ScoreMLP
from src.train_ddpm import train_ddpm
from src.sample_ddpm import sample_returns_Q_std
from src.price_options import price_vanilla_with_ddpm


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

S0 = 100.0
mu = 0.08
sigma = 0.20
r = 0.04
days_per_year = 252
dt = 1.0 / days_per_year
H_steps = 21

N_train = 1_000
T = 1000  # diffusion steps

v0 = sigma**2 * dt
s0 = math.sqrt(v0)
mP = (mu - 0.5 * sigma**2) * dt

rng = np.random.default_rng(42)
torch.manual_seed(42)
np.random.seed(42)

# THIS is the important part:
y_train = rng.normal(loc=mP, scale=s0, size=(N_train,)).astype(np.float32)
y_train_std = (y_train - mP) / s0



In [5]:
from torch.utils.data import TensorDataset, DataLoader
train_loader = DataLoader(TensorDataset(torch.from_numpy(y_train_std).unsqueeze(1)),
                          batch_size=512, shuffle=True)


In [6]:
model = ScoreMLP(hidden_dim=256, time_emb_dim=64).to(device)

model = train_ddpm(
    model=model,
    train_loader=train_loader,
    alphas_bar=alphas_bar,
    T=T,
    device=device,
    epochs=60
)

Epoch 1/60:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 2/60:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 3/60:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 4/60:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 5/60:   0%|          | 0/2 [00:00<?, ?it/s]

...

Epoch 56/60:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 57/60:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 58/60:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 59/60:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 60/60:   0%|          | 0/2 [00:00<?, ?it/s]

In [7]:
from src_v2.finance.sample_returns_shifted import sample_returns_Q_epsilon_shift
from src.experiments import martingale_trajectory, black_scholes_price, ddpm_price_from_returns

In [8]:
def run_case(apply_shift: bool, apply_mean_projection: bool):
    rr = r if apply_shift else mu  # kill dm if no shift
    returns_Q = sample_returns_Q_epsilon_shift(
        model=model,
        n_paths=20000,
        H_steps=H_steps,
        mu=mu,
        r=rr,
        sigma=sigma,
        dt=dt,
        alphas=alphas,
        alphas_bar=alphas_bar,
        betas=betas,
        s0=(sigma*math.sqrt(dt)),
        device=device,
        apply_mean_projection=apply_mean_projection,
    )
    times, M = martingale_trajectory(returns_Q, S0=S0, r=r, dt=dt)
    return float(returns_Q.mean()), float(returns_Q.std(ddof=1)), float(M[-1]), float(np.max(np.abs(M - S0)))

for shift in [False, True]:
    for proj in [False, True]:
        m, s, Mend, maxdev = run_case(shift, proj)
        print(f"shift={shift} mean_proj={proj}  mean={m:.6g} std={s:.6g}  M_end={Mend:.6g}  max|M-S0|={maxdev:.6g}")


shift=False mean_proj=False  mean=0.000465022 std=0.0119969  M_end=100.797  max|M-S0|=0.797352
shift=False mean_proj=True  mean=0.000238095 std=0.0120004  M_end=100.319  max|M-S0|=0.319032
shift=True mean_proj=False  mean=0.00011971 std=0.0119704  M_end=100.068  max|M-S0|=0.0677541
shift=True mean_proj=True  mean=7.93651e-05 std=0.0119816  M_end=99.9852  max|M-S0|=0.0621187


In [9]:
returns_Q = sample_returns_Q_epsilon_shift(
    model=model,
    n_paths=10000,
    H_steps=H_steps,
    mu=mu,
    r=r,
    sigma=sigma,
    dt=dt,
    alphas=alphas,
    alphas_bar=alphas_bar,
    betas=betas,
    s0=(sigma*math.sqrt(dt)),
    device=device,
    apply_mean_projection=False,
)

Ks = [0.8*S0, 0.9*S0, 1.0*S0, 1.1*S0, 1.2*S0]
T_total = H_steps * dt

for K in Ks:
    p_ddpm, se_ddpm = ddpm_price_from_returns(returns_Q, S0=S0, K=K, r=r, T_total=T_total, option_type="call")
    p_bs = black_scholes_price(S0=S0, K=K, T=T_total, r=r, sigma=sigma, option_type="call")
    print(f"K={K:.2f}  DDPM={p_ddpm:.6f} ± {se_ddpm:.6f}   BS={p_bs:.6f}   diff={p_ddpm-p_bs:.6f}")


K=80.00  DDPM=20.303004 ± 0.054858   BS=20.266274   diff=0.036729
K=90.00  DDPM=10.384167 ± 0.053806   BS=10.362708   diff=0.021459
K=100.00  DDPM=2.371866 ± 0.034198   BS=2.469362   diff=-0.097495
K=110.00  DDPM=0.109727 ± 0.006794   BS=0.142766   diff=-0.033039
K=120.00  DDPM=0.000285 ± 0.000264   BS=0.001686   diff=-0.001400


In [11]:
from src_v2.finance.sample_returns_shifted import sample_returns_Q_epsilon_shift
from src.price_asian_options import ddpm_asian_price_from_returns, gbm_mc_asian_price
from src.experiments import martingale_trajectory

# params
S0 = 100.0
K  = 100.0
option_type = "call"
include_S0_in_average = False

n_paths = 20000

# sample RN log-returns (shape: n_paths x H_steps)
returns_Q = sample_returns_Q_epsilon_shift(
    model=model,
    n_paths=n_paths,
    H_steps=H_steps,
    mu=mu,
    r=r,
    sigma=sigma,
    dt=dt,
    alphas=alphas,
    alphas_bar=alphas_bar,
    betas=betas,
    s0=(sigma * math.sqrt(dt)),
    device=device,
    apply_mean_projection=False,   # can toggle True
)

# martingale trajectory quick check
times, M = martingale_trajectory(returns_Q, S0=S0, r=r, dt=dt)
print("M_end:", float(M[-1]), "max|M-S0|:", float(np.max(np.abs(M - S0))))

# DDPM Asian price from sampled returns
p_ddpm, se_ddpm = ddpm_asian_price_from_returns(
    returns_Q=returns_Q,
    S0=S0,
    K=K,
    r=r,
    dt=dt,
    option_type=option_type,
    include_S0_in_average=include_S0_in_average,
)

# GBM benchmark Asian price
p_gbm, se_gbm = gbm_mc_asian_price(
    S0=S0,
    K=K,
    T=H_steps * dt,
    r=r,
    sigma=sigma,
    n_paths=n_paths,
    H_steps=H_steps,
    option_type=option_type,
    include_S0_in_average=include_S0_in_average,
)

print("Asian DDPM:", p_ddpm, "±", se_ddpm)
print("Asian GBM :", p_gbm,  "±", se_gbm)
print("diff:", p_ddpm - p_gbm)


M_end: 100.12381590121366 max|M-S0|: 0.13204905650646026
Asian DDPM: 1.427801415497206 ± 0.014379312639291653
Asian GBM : 1.4579333061492292 ± 0.01496476554460079
diff: -0.030131890652023152
